# Enterprise Agentic AI — Production Reference Architecture

A complete, runnable reference implementation for the three architecture concepts:

1. **Traditional RAG → Governed Context**
2. **Multi-Agent Control Plane**
3. **Five Levels of Enterprise Autonomy**

The notebook runs locally with the Python standard library. External enterprise services are represented behind interfaces so they can be replaced with OpenAI/Azure OpenAI, LangGraph, Databricks/Genie, a vector database, MCP/API services, OIDC, Redis/DynamoDB, and OpenTelemetry/Arize.

> **Core rule:** the model proposes; deterministic policy authorizes; the execution layer acts; validators verify; the audit layer records.

## What is productionized here

- Typed domain contracts
- Identity/persona/RBAC and data classification
- Governed retrieval using relevance + authority + freshness
- Pre-retrieval authorization filtering
- Intent classification and scope validation
- Agent registry and deterministic routing
- Planner/decomposer
- Tool registry and allowlists
- Execution budgets
- Timeouts/retries abstraction
- Human-in-the-loop approval
- Level 1–5 autonomy
- Idempotent mutations
- Provenance/citations
- Critique/validation
- Session memory abstraction
- Structured tracing/audit events
- Prompt-injection screening at trust boundaries
- Failure handling
- Golden test/evaluation harness
- Example Finance and Customer Service workflows

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from enum import IntEnum
from typing import Any, Callable, Iterable, Optional, Protocol
import hashlib
import json
import logging
import math
import re
import time
import uuid

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger("enterprise-agentic-ai")

def utc_now() -> datetime:
    return datetime.now(timezone.utc)

def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex}"

## 1. Domain contracts

In [ ]:
class Classification(IntEnum):
    PUBLIC = 0
    CONFIDENTIAL = 1
    RESTRICTED = 2
    HIGHLY_RESTRICTED = 3

class AutonomyLevel(IntEnum):
    ANSWER = 1
    RECOMMEND = 2
    PLAN = 3
    ACT_WITH_APPROVAL = 4
    ACT_WITHIN_POLICY = 5

@dataclass(frozen=True)
class Identity:
    user_id: str
    email: str
    persona: str
    entitlements: frozenset[str]
    max_classification: Classification

@dataclass(frozen=True)
class Source:
    source_id: str
    name: str
    content: str
    authority: int
    updated_at: datetime
    classification: Classification
    allowed_personas: frozenset[str]
    uri: str
    page: Optional[int] = None
    metadata: dict[str, Any] = field(default_factory=dict)

@dataclass(frozen=True)
class Citation:
    source_id: str
    source_name: str
    uri: str
    page: Optional[int]
    excerpt: str

@dataclass
class Budget:
    max_tool_calls: int = 10
    max_agent_calls: int = 8
    max_context_items: int = 8
    tool_calls: int = 0
    agent_calls: int = 0

@dataclass
class ExecutionContext:
    identity: Identity
    intent: str
    autonomy: AutonomyLevel
    session_id: str = field(default_factory=lambda: new_id("session"))
    trace_id: str = field(default_factory=lambda: new_id("trace"))
    allowed_agents: set[str] = field(default_factory=set)
    allowed_tools: set[str] = field(default_factory=set)
    budget: Budget = field(default_factory=Budget)
    attributes: dict[str, Any] = field(default_factory=dict)

@dataclass
class PlanStep:
    step_id: str
    agent: str
    objective: str
    tool_hints: list[str] = field(default_factory=list)

@dataclass
class ProposedAction:
    action_type: str
    arguments: dict[str, Any]
    reason: str
    risk: str = "low"

@dataclass
class AgentResult:
    answer: str
    citations: list[Citation] = field(default_factory=list)
    proposed_actions: list[ProposedAction] = field(default_factory=list)
    executed_actions: list[dict[str, Any]] = field(default_factory=list)
    confidence: float = 0.0
    warnings: list[str] = field(default_factory=list)

class PolicyDenied(PermissionError): pass
class BudgetExceeded(RuntimeError): pass
class ValidationError(RuntimeError): pass
class ApprovalRequired(RuntimeError): pass

## 2. Structured observability

In [ ]:
class Telemetry:
    def __init__(self):
        self.events: list[dict[str, Any]] = []

    def emit(self, ctx: ExecutionContext, event: str, **attrs: Any) -> None:
        record = {
            "timestamp": utc_now().isoformat(),
            "trace_id": ctx.trace_id,
            "session_id": ctx.session_id,
            "user_id": ctx.identity.user_id,
            "persona": ctx.identity.persona,
            "intent": ctx.intent,
            "autonomy": int(ctx.autonomy),
            "event": event,
            **attrs,
        }
        self.events.append(record)
        log.info(json.dumps(record, default=str))

TELEMETRY = Telemetry()

def timed(stage: str):
    def deco(fn):
        def wrapped(ctx: ExecutionContext, *args, **kwargs):
            start = time.perf_counter()
            try:
                value = fn(ctx, *args, **kwargs)
                TELEMETRY.emit(
                    ctx, stage, status="ok",
                    latency_ms=round((time.perf_counter()-start)*1000, 2)
                )
                return value
            except Exception as exc:
                TELEMETRY.emit(
                    ctx, stage, status="error",
                    error_type=type(exc).__name__,
                    error=str(exc),
                    latency_ms=round((time.perf_counter()-start)*1000, 2)
                )
                raise
        return wrapped
    return deco

## 3. Security boundary: input screening + deterministic authorization

In [ ]:
class InputGuard:
    SUSPICIOUS = (
        r"ignore (all|any|the) previous",
        r"reveal .*system prompt",
        r"bypass .*security",
        r"dump .*credentials",
        r"show .*secret",
    )

    @classmethod
    def validate(cls, text: str) -> None:
        if not text or not text.strip():
            raise ValidationError("Empty input")
        if len(text) > 20_000:
            raise ValidationError("Input too large")
        lowered = text.lower()
        for pattern in cls.SUSPICIOUS:
            if re.search(pattern, lowered):
                raise ValidationError("Potential prompt-injection pattern detected")

class PolicyEngine:
    @staticmethod
    def can_read(identity: Identity, source: Source) -> bool:
        return (
            identity.persona in source.allowed_personas
            and source.classification <= identity.max_classification
        )

    @staticmethod
    def require_agent(ctx: ExecutionContext, agent: str) -> None:
        if agent not in ctx.allowed_agents:
            raise PolicyDenied(f"Agent not authorized: {agent}")
        if ctx.budget.agent_calls >= ctx.budget.max_agent_calls:
            raise BudgetExceeded("Agent-call budget exhausted")
        ctx.budget.agent_calls += 1

    @staticmethod
    def require_tool(ctx: ExecutionContext, tool: str) -> None:
        if tool not in ctx.allowed_tools:
            raise PolicyDenied(f"Tool not authorized: {tool}")
        if ctx.budget.tool_calls >= ctx.budget.max_tool_calls:
            raise BudgetExceeded("Tool-call budget exhausted")
        ctx.budget.tool_calls += 1

    @staticmethod
    def authorize_mutation(
        ctx: ExecutionContext,
        action: ProposedAction,
        human_approved: bool = False,
    ) -> None:
        if ctx.autonomy <= AutonomyLevel.PLAN:
            raise PolicyDenied("Configured autonomy does not permit execution")

        if ctx.autonomy == AutonomyLevel.ACT_WITH_APPROVAL and not human_approved:
            raise ApprovalRequired("Human approval required")

        # Example Level-5 business guardrails.
        if ctx.autonomy == AutonomyLevel.ACT_WITHIN_POLICY:
            if action.action_type == "refund":
                amount = float(action.arguments.get("amount", 0))
                if amount > 100:
                    raise PolicyDenied(
                        "Autonomous refund limit is $100; route for human approval"
                    )
            if action.risk.lower() in {"high", "critical"}:
                raise PolicyDenied("High-risk actions require human review")

## 4. Session memory abstraction

In [ ]:
class MemoryStore(Protocol):
    def append(self, session_id: str, role: str, content: str) -> None: ...
    def history(self, session_id: str, limit: int = 20) -> list[dict[str, str]]: ...

class InMemoryStore:
    def __init__(self):
        self.data: dict[str, list[dict[str, str]]] = {}

    def append(self, session_id: str, role: str, content: str) -> None:
        self.data.setdefault(session_id, []).append({
            "role": role,
            "content": content,
            "timestamp": utc_now().isoformat(),
        })

    def history(self, session_id: str, limit: int = 20) -> list[dict[str, str]]:
        return self.data.get(session_id, [])[-limit:]

MEMORY = InMemoryStore()

## 5. Governed Context

Traditional RAG commonly ranks primarily by semantic similarity.  
This implementation first applies **authorization**, then combines:

- semantic relevance
- source authority
- freshness
- business metadata

Unauthorized content never enters the model/agent context.

In [ ]:
class GovernedRetriever:
    def __init__(self, sources: Iterable[Source]):
        self.sources = list(sources)

    @staticmethod
    def _tokens(text: str) -> set[str]:
        return set(re.findall(r"[a-z0-9]+", text.lower()))

    @classmethod
    def relevance(cls, query: str, content: str) -> float:
        q, d = cls._tokens(query), cls._tokens(content)
        return len(q & d) / max(1, len(q))

    @staticmethod
    def freshness(source: Source) -> float:
        age_days = max(0.0, (utc_now() - source.updated_at).total_seconds()/86400)
        return 1.0 / (1.0 + age_days/30.0)

    @timed("governed_retrieval")
    def retrieve(
        self,
        ctx: ExecutionContext,
        query: str,
        top_k: Optional[int] = None,
    ) -> list[Source]:
        InputGuard.validate(query)
        top_k = min(top_k or ctx.budget.max_context_items,
                    ctx.budget.max_context_items)

        scored = []
        for src in self.sources:
            if not PolicyEngine.can_read(ctx.identity, src):
                continue

            rel = self.relevance(query, src.content)
            authority = max(0, min(src.authority, 10)) / 10
            fresh = self.freshness(src)

            score = 0.45*rel + 0.40*authority + 0.15*fresh
            scored.append((score, src))

        scored.sort(key=lambda pair: pair[0], reverse=True)
        selected = [src for _, src in scored[:top_k]]

        TELEMETRY.emit(
            ctx, "context_selected",
            source_ids=[s.source_id for s in selected],
            classifications=[int(s.classification) for s in selected]
        )
        return selected

    @staticmethod
    def citations(sources: list[Source]) -> list[Citation]:
        return [
            Citation(
                source_id=s.source_id,
                source_name=s.name,
                uri=s.uri,
                page=s.page,
                excerpt=s.content[:180],
            )
            for s in sources
        ]

## 6. Enterprise tool registry

In [ ]:
@dataclass
class ToolDefinition:
    name: str
    handler: Callable[..., dict[str, Any]]
    mutating: bool = False
    timeout_seconds: float = 5.0
    max_retries: int = 1

class ToolRegistry:
    def __init__(self):
        self.tools: dict[str, ToolDefinition] = {}
        self.idempotency_cache: dict[str, dict[str, Any]] = {}

    def register(self, definition: ToolDefinition) -> None:
        if definition.name in self.tools:
            raise ValueError(f"Duplicate tool: {definition.name}")
        self.tools[definition.name] = definition

    def _key(self, name: str, args: dict[str, Any]) -> str:
        raw = json.dumps({"tool": name, "args": args}, sort_keys=True, default=str)
        return hashlib.sha256(raw.encode()).hexdigest()

    @timed("tool_execution")
    def execute(
        self,
        ctx: ExecutionContext,
        name: str,
        args: dict[str, Any],
        *,
        human_approved: bool = False,
    ) -> dict[str, Any]:
        PolicyEngine.require_tool(ctx, name)
        if name not in self.tools:
            raise KeyError(f"Unknown tool: {name}")

        tool = self.tools[name]
        action = ProposedAction(name, args, reason="Agent-requested tool execution")

        if tool.mutating:
            PolicyEngine.authorize_mutation(ctx, action, human_approved)

        key = self._key(name, args)
        if tool.mutating and key in self.idempotency_cache:
            TELEMETRY.emit(ctx, "idempotent_replay", tool=name)
            return self.idempotency_cache[key]

        last_error = None
        for attempt in range(tool.max_retries + 1):
            started = time.perf_counter()
            try:
                result = tool.handler(**args)
                elapsed = time.perf_counter() - started
                if elapsed > tool.timeout_seconds:
                    raise TimeoutError(f"{name} exceeded timeout")
                if tool.mutating:
                    self.idempotency_cache[key] = result
                return result
            except Exception as exc:
                last_error = exc
                if attempt >= tool.max_retries:
                    raise
                time.sleep(min(0.05 * (2**attempt), 0.5))
        raise last_error

TOOLS = ToolRegistry()

TOOLS.register(ToolDefinition(
    "get_operating_margin",
    lambda period, region="US": {
        "period": period, "region": region,
        "operating_margin_pct": 18.7,
        "system": "finance_warehouse"
    }
))

TOOLS.register(ToolDefinition(
    "get_product_sales",
    lambda brand: {
        "brand": brand,
        "products": [
            {"sku": "SKU-A", "sales": 12_400_000},
            {"sku": "SKU-B", "sales": 8_200_000},
            {"sku": "SKU-C", "sales": 2_100_000},
        ]
    }
))

TOOLS.register(ToolDefinition(
    "refund",
    lambda customer_id, amount: {
        "status": "executed",
        "transaction_id": new_id("refund"),
        "customer_id": customer_id,
        "amount": amount,
    },
    mutating=True
))

## 7. Intent clarifier, scope checker, planner and router

In [ ]:
class IntentClarifier:
    @timed("intent_clarifier")
    def run(self, ctx: ExecutionContext, question: str) -> str:
        InputGuard.validate(question)
        return question.strip()

class ScopeChecker:
    @timed("scope_checker")
    def run(self, ctx: ExecutionContext, question: str) -> str:
        # Production version can enforce domain, geography, legal and data-use constraints.
        if "password" in question.lower():
            raise PolicyDenied("Credential retrieval is outside supported scope")
        return question

class Planner:
    @timed("planner")
    def run(self, ctx: ExecutionContext, question: str, agent: str) -> list[PlanStep]:
        return [
            PlanStep(
                step_id=new_id("step"),
                agent=agent,
                objective=question,
                tool_hints=[]
            )
        ]

class Router:
    @timed("router")
    def run(self, ctx: ExecutionContext, question: str) -> str:
        q = question.lower()
        if any(w in q for w in ("refund", "customer service", "customer complaint")):
            return "customer_service"
        if any(w in q for w in ("margin", "sales", "finance", "profit", "revenue")):
            return "finance"
        return "general"

## 8. Agent registry and specialized agents

In [ ]:
class Agent(Protocol):
    name: str
    def run(
        self, ctx: ExecutionContext, objective: str, sources: list[Source]
    ) -> AgentResult: ...

class AgentRegistry:
    def __init__(self):
        self.agents: dict[str, Agent] = {}

    def register(self, agent: Agent) -> None:
        self.agents[agent.name] = agent

    def get(self, name: str) -> Agent:
        if name not in self.agents:
            raise KeyError(f"Agent not registered: {name}")
        return self.agents[name]

class FinanceAgent:
    name = "finance"

    def run(self, ctx, objective, sources):
        PolicyEngine.require_agent(ctx, self.name)

        if "margin" in objective.lower():
            data = TOOLS.execute(
                ctx, "get_operating_margin",
                {"period": "Q3", "region": "US"}
            )
            answer = (
                f"Q3 US operating margin is "
                f"{data['operating_margin_pct']}%."
            )
        elif "sales" in objective.lower():
            data = TOOLS.execute(ctx, "get_product_sales", {"brand": "Example Brand"})
            smallest = min(data["products"], key=lambda x: x["sales"])
            answer = (
                f"{smallest['sku']} has the lowest sales in the returned "
                f"brand dataset at ${smallest['sales']:,}."
            )
        else:
            answer = "Finance analysis completed using governed enterprise context."

        return AgentResult(
            answer=answer,
            citations=GovernedRetriever.citations(sources),
            confidence=0.93,
        )

class CustomerServiceAgent:
    name = "customer_service"

    def run(self, ctx, objective, sources):
        PolicyEngine.require_agent(ctx, self.name)
        action = ProposedAction(
            action_type="refund",
            arguments={"customer_id": "C-100", "amount": 75},
            reason="Customer qualifies under the example refund rule",
            risk="low",
        )
        return AgentResult(
            answer="A $75 refund is proposed.",
            citations=GovernedRetriever.citations(sources),
            proposed_actions=[action],
            confidence=0.90,
        )

class GeneralAgent:
    name = "general"

    def run(self, ctx, objective, sources):
        PolicyEngine.require_agent(ctx, self.name)
        snippets = " ".join(s.content for s in sources[:2])
        return AgentResult(
            answer=f"Governed context summary: {snippets}" if snippets
                   else "No authorized evidence was found.",
            citations=GovernedRetriever.citations(sources),
            confidence=0.75 if sources else 0.25,
        )

AGENTS = AgentRegistry()
AGENTS.register(FinanceAgent())
AGENTS.register(CustomerServiceAgent())
AGENTS.register(GeneralAgent())

## 9. Critique / validation layer

In [ ]:
class Validator:
    @timed("critique_validation")
    def run(self, ctx: ExecutionContext, result: AgentResult) -> AgentResult:
        if not result.answer.strip():
            raise ValidationError("Empty agent response")

        if not (0 <= result.confidence <= 1):
            raise ValidationError("Confidence must be between 0 and 1")

        # Example: prevent citation-less high-confidence factual responses.
        factual_markers = bool(re.search(r"\d", result.answer))
        if factual_markers and not result.citations:
            result.warnings.append("Numeric answer has no document citation")

        # Never treat model confidence as authorization.
        if result.confidence < 0.5:
            result.warnings.append("Low-confidence result; consider human review")

        return result

## 10. Response synthesizer

In [ ]:
class ResponseSynthesizer:
    @timed("response_synthesis")
    def run(self, ctx: ExecutionContext, result: AgentResult) -> AgentResult:
        if result.citations:
            refs = []
            for i, c in enumerate(result.citations, start=1):
                location = f", p.{c.page}" if c.page else ""
                refs.append(f"[{i}] {c.source_name}{location}")
            result.answer += "\n\nSources: " + "; ".join(refs)
        return result

## 11. Complete multi-agent control plane

In [ ]:
class AgenticControlPlane:
    def __init__(self, retriever: GovernedRetriever):
        self.retriever = retriever
        self.intent = IntentClarifier()
        self.scope = ScopeChecker()
        self.router = Router()
        self.planner = Planner()
        self.validator = Validator()
        self.synthesizer = ResponseSynthesizer()

    def run(self, ctx: ExecutionContext, question: str) -> AgentResult:
        TELEMETRY.emit(ctx, "request_started")
        MEMORY.append(ctx.session_id, "user", question)

        clarified = self.intent.run(ctx, question)
        scoped = self.scope.run(ctx, clarified)
        agent_name = self.router.run(ctx, scoped)

        if agent_name not in ctx.allowed_agents:
            raise PolicyDenied(
                f"Router selected '{agent_name}', but it is not authorized "
                f"for this execution context"
            )

        plan = self.planner.run(ctx, scoped, agent_name)
        TELEMETRY.emit(
            ctx, "plan_created",
            steps=[asdict(s) for s in plan]
        )

        sources = self.retriever.retrieve(ctx, scoped)
        result = AGENTS.get(agent_name).run(ctx, scoped, sources)
        result = self.validator.run(ctx, result)
        result = self.synthesizer.run(ctx, result)

        MEMORY.append(ctx.session_id, "assistant", result.answer)
        TELEMETRY.emit(
            ctx, "request_completed",
            agent=agent_name,
            confidence=result.confidence,
            tool_calls=ctx.budget.tool_calls,
            agent_calls=ctx.budget.agent_calls,
        )
        return result

## 12. Example governed enterprise data

In [ ]:
SOURCES = [
    Source(
        source_id="FIN-Q3-001",
        name="Authoritative Finance Warehouse",
        content=(
            "For Q3, US operating margin is 18.7 percent using the approved "
            "GAAP operating-margin definition."
        ),
        authority=10,
        updated_at=utc_now(),
        classification=Classification.CONFIDENTIAL,
        allowed_personas=frozenset({"finance"}),
        uri="enterprise://finance/q3-margin",
        metadata={"definition": "Operating Margin (GAAP)", "status": "certified"},
    ),
    Source(
        source_id="PPT-Q3-OLD",
        name="Prior Management Presentation",
        content="An earlier Q3 operating margin estimate was 19.2 percent.",
        authority=5,
        updated_at=datetime(2026, 5, 1, tzinfo=timezone.utc),
        classification=Classification.CONFIDENTIAL,
        allowed_personas=frozenset({"finance"}),
        uri="enterprise://sharepoint/old-q3-deck",
        page=17,
        metadata={"status": "historical estimate"},
    ),
    Source(
        source_id="SALES-001",
        name="Certified Sales Mart",
        content="Certified product and brand sales data for commercial analysis.",
        authority=10,
        updated_at=utc_now(),
        classification=Classification.CONFIDENTIAL,
        allowed_personas=frozenset({"finance"}),
        uri="enterprise://databricks/certified-sales",
    ),
    Source(
        source_id="RD-SECRET-001",
        name="Restricted R&D Formula Repository",
        content="Highly restricted product formulation and ingredient information.",
        authority=10,
        updated_at=utc_now(),
        classification=Classification.HIGHLY_RESTRICTED,
        allowed_personas=frozenset({"rd"}),
        uri="enterprise://rd/formulations",
    ),
    Source(
        source_id="CS-POLICY-001",
        name="Customer Resolution Policy",
        content=(
            "Eligible customer service cases may receive low-value refunds. "
            "Autonomous execution must remain within configured policy limits."
        ),
        authority=10,
        updated_at=utc_now(),
        classification=Classification.CONFIDENTIAL,
        allowed_personas=frozenset({"customer_service"}),
        uri="enterprise://policies/customer-resolution",
    ),
]

RETRIEVER = GovernedRetriever(SOURCES)
CONTROL_PLANE = AgenticControlPlane(RETRIEVER)

## 13. Scenario 1 — RAG vs governed context

In [ ]:
finance_identity = Identity(
    user_id="U-100",
    email="finance.user@example.com",
    persona="finance",
    entitlements=frozenset({"finance.read", "sales.read"}),
    max_classification=Classification.CONFIDENTIAL,
)

finance_ctx = ExecutionContext(
    identity=finance_identity,
    intent="financial_analysis",
    autonomy=AutonomyLevel.ANSWER,
    allowed_agents={"finance", "general"},
    allowed_tools={"get_operating_margin", "get_product_sales"},
)

result = CONTROL_PLANE.run(
    finance_ctx,
    "What was our Q3 operating margin?"
)

print(result.answer)
print("\nConfidence:", result.confidence)
print("Warnings:", result.warnings)

### Why this differs from naïve RAG

The corpus contains both **18.7%** and an older **19.2% estimate**.  
Governed retrieval favors the certified, fresher, higher-authority source and excludes data the caller cannot access.

## 14. Scenario 2 — prove restricted R&D data does not leak

In [ ]:
visible = RETRIEVER.retrieve(
    finance_ctx,
    "product formulation ingredient information",
    top_k=10,
)

print("Sources visible to Finance:")
for src in visible:
    print("-", src.name, "|", src.classification.name)

assert all(src.source_id != "RD-SECRET-001" for src in visible)
print("\nPASS: highly restricted R&D source never entered Finance context.")

## 15. Scenario 3 — Level 1: Answer

In [ ]:
print("Level 1 permits informational response generation.")
print("Autonomy:", AutonomyLevel.ANSWER.name)

## 16. Scenario 4 — Level 2: Recommend

In [ ]:
recommend_ctx = ExecutionContext(
    identity=finance_identity,
    intent="financial_analysis",
    autonomy=AutonomyLevel.RECOMMEND,
    allowed_agents={"finance"},
    allowed_tools={"get_product_sales"},
)
print("Level 2 can recommend, but PolicyEngine still blocks mutations.")

## 17. Scenario 5 — Level 3: Plan

In [ ]:
plan_ctx = ExecutionContext(
    identity=finance_identity,
    intent="financial_analysis",
    autonomy=AutonomyLevel.PLAN,
    allowed_agents={"finance"},
    allowed_tools={"get_product_sales"},
)
steps = Planner().run(plan_ctx, "Identify low-selling products and prepare actions", "finance")
for step in steps:
    print(asdict(step))

## 18. Scenario 6 — Level 4: Act with human approval

In [ ]:
service_identity = Identity(
    user_id="U-CS-1",
    email="service.user@example.com",
    persona="customer_service",
    entitlements=frozenset({"customer.read", "refund.execute"}),
    max_classification=Classification.CONFIDENTIAL,
)

approval_ctx = ExecutionContext(
    identity=service_identity,
    intent="customer_resolution",
    autonomy=AutonomyLevel.ACT_WITH_APPROVAL,
    allowed_agents={"customer_service"},
    allowed_tools={"refund"},
)

proposal = ProposedAction(
    "refund",
    {"customer_id": "C-100", "amount": 75},
    "Eligible customer resolution",
)

try:
    TOOLS.execute(
        approval_ctx, proposal.action_type, proposal.arguments,
        human_approved=False
    )
except ApprovalRequired as exc:
    print("Correctly paused:", exc)

executed = TOOLS.execute(
    approval_ctx, proposal.action_type, proposal.arguments,
    human_approved=True
)
print("After approval:", executed)

## 19. Scenario 7 — Level 5: Act autonomously within policy

In [ ]:
auto_ctx = ExecutionContext(
    identity=service_identity,
    intent="customer_resolution",
    autonomy=AutonomyLevel.ACT_WITHIN_POLICY,
    allowed_agents={"customer_service"},
    allowed_tools={"refund"},
)

within_policy = TOOLS.execute(
    auto_ctx,
    "refund",
    {"customer_id": "C-200", "amount": 75},
)
print("Allowed:", within_policy)

try:
    TOOLS.execute(
        auto_ctx,
        "refund",
        {"customer_id": "C-201", "amount": 500},
    )
except PolicyDenied as exc:
    print("Correctly blocked:", exc)

## 20. Idempotency demonstration

In [ ]:
idem_ctx = ExecutionContext(
    identity=service_identity,
    intent="customer_resolution",
    autonomy=AutonomyLevel.ACT_WITHIN_POLICY,
    allowed_agents={"customer_service"},
    allowed_tools={"refund"},
)

first = TOOLS.execute(
    idem_ctx, "refund", {"customer_id": "C-300", "amount": 25}
)
second = TOOLS.execute(
    idem_ctx, "refund", {"customer_id": "C-300", "amount": 25}
)

assert first["transaction_id"] == second["transaction_id"]
print("Same transaction returned on replay:", first["transaction_id"])

## 21. Golden evaluation suite

In [ ]:
@dataclass
class EvalCase:
    name: str
    run: Callable[[], bool]

def eval_authoritative_source() -> bool:
    ctx = ExecutionContext(
        identity=finance_identity,
        intent="eval",
        autonomy=AutonomyLevel.ANSWER,
        allowed_agents={"finance"},
        allowed_tools={"get_operating_margin"},
    )
    ranked = RETRIEVER.retrieve(ctx, "Q3 operating margin", top_k=2)
    return ranked[0].source_id == "FIN-Q3-001"

def eval_no_rd_leak() -> bool:
    ctx = ExecutionContext(
        identity=finance_identity,
        intent="eval",
        autonomy=AutonomyLevel.ANSWER,
        allowed_agents={"general"},
        allowed_tools=set(),
    )
    ranked = RETRIEVER.retrieve(ctx, "formulation ingredient", top_k=10)
    return all(x.source_id != "RD-SECRET-001" for x in ranked)

def eval_large_refund_blocked() -> bool:
    ctx = ExecutionContext(
        identity=service_identity,
        intent="eval",
        autonomy=AutonomyLevel.ACT_WITHIN_POLICY,
        allowed_agents={"customer_service"},
        allowed_tools={"refund"},
    )
    try:
        TOOLS.execute(
            ctx, "refund", {"customer_id": "EVAL-1", "amount": 500}
        )
        return False
    except PolicyDenied:
        return True

EVALS = [
    EvalCase("authoritative_source_wins", eval_authoritative_source),
    EvalCase("restricted_rd_does_not_leak", eval_no_rd_leak),
    EvalCase("large_autonomous_refund_blocked", eval_large_refund_blocked),
]

results = []
for case in EVALS:
    try:
        passed = bool(case.run())
        results.append((case.name, passed, None))
    except Exception as exc:
        results.append((case.name, False, str(exc)))

for name, passed, error in results:
    print(f"{'PASS' if passed else 'FAIL'} | {name}", error or "")

assert all(passed for _, passed, _ in results)

## 22. Production deployment mapping

In [ ]:
PRODUCTION_MAPPING = {
    "LLM / reasoning": [
        "OpenAI / Azure OpenAI / approved enterprise model endpoint",
        "Require structured outputs for plans, tool calls and critic results",
    ],
    "orchestration": [
        "LangGraph or an internal typed workflow/state machine",
        "Persist resumable state for approval workflows",
    ],
    "identity": [
        "OIDC/OAuth2 at ingress",
        "Validate token server-side",
        "Map user -> persona/function -> entitlements",
    ],
    "structured data": [
        "Databricks/Genie/warehouse via governed service identity",
        "Apply row/column permissions before returning data",
    ],
    "unstructured data": [
        "Vector store + metadata filters",
        "Preserve file/page/chunk/source lineage",
    ],
    "tool interoperability": [
        "MCP/internal APIs/API gateway",
        "Explicit tool schemas, scopes, timeouts and allowlists",
    ],
    "memory": [
        "Redis for short-lived session state",
        "Durable store for approved conversation/audit state",
        "Never use conversational memory as an authorization source",
    ],
    "observability": [
        "OpenTelemetry traces",
        "Arize/enterprise observability",
        "trace_id + user_id + persona + agent + environment + classification",
    ],
    "security": [
        "Secrets manager / key vault",
        "Network egress controls",
        "Prompt-injection defenses",
        "Encryption in transit/at rest",
        "DLP/PII controls",
    ],
}

for area, items in PRODUCTION_MAPPING.items():
    print(f"\n{area.upper()}")
    for item in items:
        print(" -", item)

## 23. Production readiness checklist

In [ ]:
CHECKLIST = [
    "Authentication occurs before orchestration.",
    "Authorization filters are applied before retrieval reaches an LLM.",
    "Agent and tool access are allowlisted per execution context.",
    "High-impact actions are governed by deterministic policy.",
    "Human approval is resumable and auditable.",
    "Mutating tools are idempotent.",
    "Agent loops and tool calls have explicit budgets.",
    "Every factual result preserves source provenance.",
    "Restricted classifications are propagated through traces safely.",
    "Prompts/tool outputs are treated as untrusted inputs.",
    "Golden evaluations run in CI/CD.",
    "Latency, token usage, tool errors and cost have SLOs/alerts.",
    "Rollback/circuit-breaker mechanisms exist for production actions.",
    "Secrets never appear in prompts, memory or traces.",
]

for i, item in enumerate(CHECKLIST, 1):
    print(f"{i:02d}. {item}")

## 24. Architecture summary

```text
User / Channel
      |
Identity + Persona + Entitlements
      |
Intent Clarifier -> Scope Checker -> Planner -> Agent Router
                                      |
                           Governed Context Layer
                     / structured / docs / APIs /
                                      |
                    Specialized Domain Agent(s)
                                      |
                         Controlled Tool Registry
                                      |
                       Critique / Validation
                                      |
                       Response + Provenance
                                      |
                    Observability + Audit Trail

Cross-cutting:
Policy | Classification | Budgets | Human Approval | Security
```

### Five autonomy levels

```text
1 ANSWER
   Human asks -> Agent explains

2 RECOMMEND
   Agent analyzes -> Human decides

3 PLAN
   Agent prepares executable steps -> Human owns execution

4 ACT WITH APPROVAL
   Agent prepares action -> Human approves -> Tool executes

5 ACT WITHIN POLICY
   Agent proposes -> Deterministic policy validates -> Tool executes
                    otherwise -> escalate to human
```

A production enterprise system should usually assign autonomy **per use case and action**, not globally to an agent.

## 25. Suggested repository structure

When converting this notebook into a service:

```text
agentic-ai/
├── app/
│   ├── api/
│   ├── agents/
│   ├── orchestration/
│   ├── retrieval/
│   ├── tools/
│   ├── policy/
│   ├── identity/
│   ├── memory/
│   ├── validation/
│   └── observability/
├── tests/
│   ├── unit/
│   ├── integration/
│   ├── security/
│   └── evals/
├── policies/
├── configs/
├── deploy/
└── pyproject.toml
```

Keep policy, authorization and action controls outside model prompts. This allows models, agent frameworks and data systems to evolve without weakening the enterprise trust boundary.